# 3 · Automated P–T Pseudosections with `psexplorer`

**Module 2 · P–T Pseudosections, part 2**

This morning's session covered building a pseudosection
interactively with `ptbuilder`. This notebook covers the other half of
`pypsbuilder`: **`psexplorer`**, its scripting layer for post-processing an
*already-built* pseudosection project — no Qt GUI needed, everything is
plain Python you can drop into any script or notebook.

We use `data/avg_pelite/tc/avgp.ptb` — a real P–T pseudosection (600–720 °C,
5–12 kbar) built from exactly the avgpelite bulk composition you formatted
for THERMOCALC/MAGEMin in `02_petropandas.ipynb`. It has already been
gridded by the instructor (a 60×60 THERMOCALC composition grid, fully
solved), so every `isopleths()`/`show_data()` call below runs instantly —
no multi-minute wait. (`avgp2.ptb`/`avgp3.ptb` in the same folder are the
same pseudosection topology *without* a grid yet, in case you want to try
`pt.calculate_composition()` yourself after class — it takes a few
minutes.)

This is the payoff of the whole day: raw microprobe data → mineral formulas
→ bulk composition → **P–T path**.


## 3.1 Environment setup and data loading

`PTPS` is `psexplorer`'s class for a P–T pseudosection project (`TXPS`/`PXPS`
are the T–X/P–X equivalents, identical API). Constructing it opens the
project file *and* re-connects to the THERMOCALC working directory it was
built in (`data/avg_pelite/tc/`), which is where the actual scriptfile,
dataset and a-x files live.


In [ ]:
from pypsbuilder import PTPS

# PTPS opens the project file *and* reconnects to the THERMOCALC working
# directory it was built in (data/avg_pelite/tc/) -- the workshop's THERMOCALC
# installation (from this morning's Installation section) needs to be present for that.
ptb_path = "../data/avg_pelite/tc/avgp.ptb"
pt = PTPS(ptb_path)

if not pt.gridded:
    pt.calculate_composition()

## 3.2 Parsing the phase diagram

`pt.show()` draws the pseudosection: every divariant field shaded by
variance, univariant lines as its boundaries, invariant points where several
lines meet.


In [ ]:
pt.show()


In [ ]:
pt.show(cmap="viridis", bulk=True, label=True, skiplabels=0.3, labelfs=8)

`pt.identify(T, p)` returns the **key** — a `frozenset` of phase
abbreviations — identifying the stable assemblage (the divariant field) at a
given point. Every data-access method in `psexplorer` is scoped by one of
these keys.


In [ ]:
key = pt.identify(650, 8)
print(f"stable assemblage at T=650 degC, p=8 kbar: {list(key)}")


The phases here are THERMOCALC's short names for the minerals in this system:
`q` quartz, `mu` muscovite, `bi` biotite, `g` garnet, `st` staurolite,
`ky`/`sill` kyanite/sillimanite, `pl4tr` plagioclase, `pa`
paragonite, `ilmm` ilmenite, `ru` rutile, `liq` melt, `H2O` water. You'll see
these same abbreviations used as the `key` everywhere from here on.


Calling `show_data(key, phase)` **without** an expression is `psexplorer`'s
way of telling you what's available to query for that phase — every site
fraction, end-member and thermodynamic property THERMOCALC calculated for
garnet (`'g'`) in this field.

Quick note: the next cell's printed line *`Missing expression argument.`* is
**not an error** — it's `psexplorer` prompting you to name what to query, and
helpfully listing everything available for that phase.

In [ ]:
pt.show_data(key, "g")


## 3.3 Extracting mineral isopleths

`pt.isopleths(phase, expr)` contours the value of `expr` for `phase` across
the whole diagram — computed separately per divariant field, so sharp jumps
across a field boundary are preserved rather than smoothed away.

`xFeX` and `xCaX` are garnet's Fe and Ca site fractions on the X site — the
compositional variables classically used for Fe–Mg exchange thermometry
(`xFeX`) and Ca-in-garnet geobarometry (`xCaX`).


In [ ]:
pt.isopleths("g", "xFeX", N=14)


In [ ]:
pt.isopleths("g", "xCaX", N=14)


Modal proportions contour the same way — here, garnet's volume mode across
the diagram:


In [ ]:
pt.isopleths("g", "mode", N=10)


`pt.show_grid(phase, expr=...)` shows the same kind of data a different way:
a plain pixel image (`imshow`) of the raw 60×60 composition grid, with **no**
per-field contouring — useful as a quick sanity check against `isopleths()`,
which is field-aware and therefore preserves sharp jumps across a
univariant line that a naive grid image blurs together.


In [ ]:
pt.show_grid("g", expr="xCaX")


## 3.4 Interactive map and P–T path evaluation

`show_data(key, phase, expr)` scatters the raw calculated values (invariant
points + univariant lines + grid points) for one field — useful to sanity-
check what `isopleths()` is interpolating from.


In [ ]:
pt.show_data(key, "g", "xCaX")


A **P–T path** is a sequence of points a rock is hypothesised to have
followed (e.g. prograde burial then exhumation). `collect_ptpath` walks an
interpolated path through the gridded pseudosection, running a real THERMOCALC
calculation at each step — that's why the cell below shows a progress bar while it runs.


In [ ]:
tpath = [610, 650, 690, 690, 670, 640]
ppath = [6.5, 9, 10, 7.5, 6, 5.5]

pa = pt.collect_ptpath(tpath, ppath, N=30)


In [ ]:
pt.show_path_modes(pa, exclude=["H2O"])


In [ ]:
pt.show_path_data(pa, "g", "mode")


For a one-off question ("what's stable at exactly this point?") without
building a whole path, `pointcalc` runs a single ad hoc THERMOCALC
calculation:


In [ ]:
result = pt.pointcalc(t=650, p=8)
print(result["g"])


## 3.5 Hands-on capstone exercise: peak P–T from garnet isopleth intersection

In `02_petropandas.ipynb` you recalculated the **rim** analysis of the real
garnet traverse from `microprobe_data.xlsx` (`Grt-profile` sheet)
— the composition closest to equilibrium with the matrix at
peak metamorphic conditions. Here we turn that single analysis into a
**peak P–T estimate**, by finding where its composition values are
simultaneously matched on the pseudosection.

**Step 1 — recompute the rim garnet's site fractions**, the same way as
Notebook 2: read the sheet, take the rim rows, and find ranges for the site
occupancies for garnet's `X` site (we will use `TC_g` model).


In [ ]:
from petropandas import pd, Grt, ProfilePlot
from petropandas.hpxeos.metapelite import TC_g

# Common display options (optional)
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.3f}'.format)

grt_profile = pd.read_excel("../data/avg_pelite/microprobe_data.xlsx", sheet_name="Grt-profile")
grt_profile

HPx-eos (`petropandas.hpxeos`) are thermodynamic models for minerals used in thermodynamic
software like THERMOCALC or MAGEMin. In petropandas we can use those models to recalculate
mineral (e.g. garnet) compositions in terms of compositional variables and directly comapre
with software outputs.

In [ ]:
grt_profile_vars = grt_profile.mineral.variables(TC_g)
grt_profile_vars


To find value ranges, we can use `describe` method:

In [ ]:
desc_rim = grt_profile_vars[grt_profile["Zone"] == "rim"].describe().round(3)
desc_rim

In [ ]:
p = ProfilePlot(columns=["xFeX", "xMgX", "xCaX", "xMnX"], split="auto")
p.add(grt_profile_vars, lw=2)
p.show()


**Step 2 — overlay isopleth bands.** `pt.overlap_isopleths` takes
repeating `(phase, expr, (low, high))` triples and shades where *all* of them
are simultaneously satisfied — exactly an isopleth-intersection thermobarometer.

The isopleth values for `rim` could be determined from calculated ranges (`min` and `max` in `desc_rim` variable):


In [ ]:
pt.overlap_isopleths(
    "g", "xFeX", (desc_rim["xFeX"]["min"], desc_rim["xFeX"]["max"]),
    "g", "xCaX", (desc_rim["xCaX"]["min"], desc_rim["xCaX"]["max"]),
    "g", "xMnX", (desc_rim["xMnX"]["min"], desc_rim["xMnX"]["max"]),
    "g", "xMgX", (desc_rim["xMgX"]["min"], desc_rim["xMgX"]["max"]),
)


The overlapping patch sits at roughly **T ≈ 660–700 °C, p ≈ 9.5–10.5 kbar**
— your estimate of the peak metamorphic conditions recorded by this
garnet's rim, derived entirely from a real microprobe analysis and the
pseudosection built from this same rock's bulk composition.

**Try it yourself:** repeat the lst step using the **core** analysis
instead of the rim — how different is the estimated P–T, and what does
that difference tell you about this garnet's growth history?


In [ ]:
# Your code here
desc_core = ...


<details><summary><b>Solution (click to expand)</b></summary>

Re-run Step 1 with the **core** composition (taking range of first 5 rows) and feed those values to the `overlap_isopleths` call:

```python
desc_core = grt_profile_vars[grt_profile["Zone"] == "core"].describe().round(3)
pt.overlap_isopleths(
    "g", "xFeX", (desc_core["xFeX"]["min"], desc_core["xFeX"]["max"]),
    "g", "xCaX", (desc_core["xCaX"]["min"], desc_core["xCaX"]["max"]),
    "g", "xMnX", (desc_core["xMnX"]["min"], desc_core["xMnX"]["max"]),
    "g", "xMgX", (desc_core["xMgX"]["min"], desc_core["xMgX"]["max"]),
)
```

**Expected result:** the core of this garnet has significantly higher xMnX and
lower xMgX comapring to the the rim). The intersection collapses to a patch
sitting at **p ≈ 6–6.5 kbar** at lower T (≈ 600–610 °C,
decidedly **lower pressure** than the rim's 9.6–10.1 kbar.

**What the core→rim difference means:** the crystal grew along a path going
from lower-pressure conditions (entering near the bottom of the diagram) to
the higher-pressure/temperature conditions recorded at the rim — a *prograde*
burial path.
</details>


## Recap

- `PTPS(projfile)` opens a built pseudosection project and reconnects it to
  its THERMOCALC working directory.
- `pt.identify(T, p)` returns the stable-assemblage `key`; `pt.show_data(
  key, phase)` lists what's queryable for a phase in that field.
- `pt.isopleths(phase, expr)` contours a compositional variable or mode
  across the whole diagram, per divariant field.
- `pt.collect_ptpath(tpath, ppath)` + `pt.show_path_modes`/`show_path_data`
  evaluate a hypothesised P–T path through the pseudosection.
- `pt.overlap_isopleths(...)` intersects several isopleth bands at once —
  the standard way to turn a measured mineral composition into a P–T
  estimate.

This closes the loop that started this morning: **raw microprobe data
(Notebook 1) → mineral formulas and bulk composition (Notebook 2) → peak
P–T conditions (this notebook)** — the same workflow you would use on your
own samples.
